<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/create_lags.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


In [2]:
%%writefile /content/drive/MyDrive/ml_project/create_lags.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

target_dir = "/content/drive/MyDrive/ml_project"
input_pickle = os.path.join(target_dir, "df_date_features.pkl")
choice_file = os.path.join(target_dir, "lags_method.txt")
output_pickle = os.path.join(target_dir, "df_lags.pkl")

# ---------------------------------------------------------
# create_lags_ui
# ---------------------------------------------------------
def create_lags_ui():

    if not os.path.exists(input_pickle):
        raise FileNotFoundError("df_date_features.pkl not found")

    df = pd.read_pickle(input_pickle)

    label_col = widgets.Label("Select the column for generating lag features:")
    dropdown = widgets.Dropdown(options=df.columns.tolist())

    label_lags = widgets.Label("Enter lag values (comma-separated), e.g. 1,2,7:")
    text_lags = widgets.Text(placeholder="1,2,7")

    btn = widgets.Button(description="Confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        col = dropdown.value
        lags_raw = text_lags.value.strip()

        with open(choice_file, "w") as f:
            f.write(col + "\n")
            f.write(lags_raw + "\n")

        with out:
            print("Saved:", choice_file)
            print("Selected column:", col)
            print("Lag values:", lags_raw)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_col,
        dropdown,
        label_lags,
        text_lags,
        btn,
        out
    ]))


# ---------------------------------------------------------
# create_lags
# ---------------------------------------------------------
def create_lags():

    if not os.path.exists(input_pickle):
        raise FileNotFoundError("df_date_features.pkl not found")

    if not os.path.exists(choice_file):
        raise FileNotFoundError("lags_method.txt not found")

    df = pd.read_pickle(input_pickle)

    with open(choice_file, "r") as f:
        lines = f.read().strip().split("\n")

    col = lines[0]
    lags_raw = lines[1]

    # parse lag values
    lags = []
    for x in lags_raw.split(","):
        x = x.strip()
        if x.isdigit():
            lags.append(int(x))

    # generate lag features
    for lag in lags:
        df[f"{col}_lag_{lag}"] = df[col].shift(lag)

    df.to_pickle(output_pickle)

    print("Saved:", output_pickle)
    print("Generated lag feature columns.")

    return df


Overwriting /content/drive/MyDrive/ml_project/create_lags.py
